# Relationship between hypertension and diabetes health insurance

The Diabetes prediction dataset is a collection of medical and demographic data from patients, along with their diabetes status (positive or negative). The data includes features such as age, gender, body mass index (BMI), hypertension, heart disease, smoking history, HbA1c level, and blood glucose level. This dataset can be used to build machine learning models to predict diabetes in patients based on their medical history and demographic information. This can be useful for healthcare professionals in identifying patients who may be at risk of developing diabetes and in developing personalized treatment plans. Additionally, the dataset can be used by researchers to explore the relationships between various medical and demographic factors and the likelihood of developing diabetes (quantity:10^5, acc:97%).

1. Hypertension is a medical condition in which the blood pressure in the arteries is persistently elevated. It has values a 0 or 1 where 0 indicates they don’t have hypertension and for 1 it means they have hypertension.
2. Diabetes is the target variable being predicted, with values of 1 indicating the presence of diabetes and 0 indicating the absence of diabetes.
3. https://www.kaggle.com/datasets/iammustafatz/diabetes-prediction-dataset

### Import necessary packages

In [1]:
import os
import numpy as np
import pandas as pd
# 💡 完美对齐 12 参数规范：直接从外部工具箱导入标准的 analyze_dataset 和 plot_cpp，本地绝不重复声明
from FL_cpp_method import analyze_dataset, plot_cpp

In [2]:
# %%
def dirichlet_logistic_regression_allocation(X, Y, Yhat, alpha_dir, num_clients, min_samples_per_client=500):
    """
    针对逻辑回归任务的 Dirichlet 非独立同分布联邦切分算法（同步对齐 OLS 稳健版）。
    将保底样本量完美对齐至 500，在完全释放极致非独立同分布异质性的同时，
    天然保障最低比例扫频时满足 n > d 与双类别并存的满秩要求，彻底免疫卡壳。
    """
    classes = np.unique(Y)
    num_classes = len(classes)
    
    # 建立类别到原始全局索引的映射并打乱
    class_indices = {c: np.where(Y == c)[0] for c in classes}
    for c in classes:
        np.random.shuffle(class_indices[c])
        
    client_indices = [[] for _ in range(num_clients)]
    
    # --------------------------------------------------------------------------
    # 阶段一：各客户端总样本量硬保障分配（各类均分保底）
    # --------------------------------------------------------------------------
    base_per_class = min_samples_per_client // num_classes
    if base_per_class < 1: base_per_class = 1
        
    for c in classes:
        indices = class_indices[c]
        available = len(indices)
        required = base_per_class * num_clients
        actual_base = base_per_class if required <= available else available // num_clients
        
        if actual_base > 0:
            for i in range(num_clients):
                client_indices[i].extend(indices[i * actual_base : (i + 1) * actual_base])
            class_indices[c] = indices[num_clients * actual_base:]
            
    # --------------------------------------------------------------------------
    # 阶段二：海量剩余资产全面交由 alpha_Dir 驱动高异质性分发
    # --------------------------------------------------------------------------
    for c in classes:
        indices = class_indices[c]
        if len(indices) == 0: continue
            
        proportions = np.random.dirichlet([alpha_dir] * num_clients)
        counts = np.floor(proportions * len(indices)).astype(int)
        
        remainder = len(indices) - np.sum(counts)
        for _ in range(remainder):
            counts[np.random.choice(num_clients)] += 1
            
        start = 0
        for i in range(num_clients):
            end = start + counts[i]
            client_indices[i].extend(indices[start:end])
            start = end
            
    # 拼合、打乱并组装最终的重排索引大数组
    reordered_indices = []
    actual_sizes = []
    for i in range(num_clients):
        np.random.shuffle(client_indices[i])
        reordered_indices.extend(client_indices[i])
        actual_sizes.append(len(client_indices[i]))
        
    reordered_indices = np.array(reordered_indices, dtype=int)
    
    # 🛠️ 核心修复：同步对 X, Y, Yhat 执行无损重排切片并返回
    return X[reordered_indices], Y[reordered_indices], Yhat[reordered_indices], actual_sizes

In [3]:
# %%
# ==============================================================================
# 1. 实验控制元参数初始化（高血压与糖尿病逻辑回归推断：alpha=0.05 对应 95% 置信区间）
# ==============================================================================
dataset_name = 'diabetes-hypertension'
alpha = 0.05  
method = "logistic"  
num_clients = 20  

xlim = [-1.0, 2.5]  
ylim = [0, 1.0]

iid_total_records = []
non_iid_total_records = []
acc_steps = np.arange(0.1, 1.1, 0.1)

# ==============================================================================
# 2. 纵向多精度大循环核心
# ==============================================================================
for acc in acc_steps:
    acc_str = f"{acc:.1f}"
    data_path = f'../data/{dataset_name}/{dataset_name}_acc_{acc_str}.npz'
    
    if not os.path.exists(data_path):
        print(f"⚠️ [跳过] 未检测到精度阶梯文件: {data_path}")
        continue
        
    print(f"\n⚡ [当前进度] 正在全面计算精度级别 -> Acc = {acc_str}")
    data = np.load(data_path)
    
    Y_total = data["Y"]
    Yhat_total = data["Y_hat"] if "Y_hat" in data.files else data["Yhat"]
    
    # --------------------------------------------------------------------------
    # 🛠️ 终极数值保护层：自适应特征标准化 (Z-score Scaling) + 噪声抗奇异双保险
    # --------------------------------------------------------------------------
    X_raw = data["X"].copy().astype(float)
    for col in range(X_raw.shape[1]):
        col_std = X_raw[:, col].std()
        if col_std > 1e-2: 
            X_raw[:, col] = (X_raw[:, col] - X_raw[:, col].mean()) / col_std
            
    X_total_safe = X_raw + np.random.normal(0, 1e-3, X_raw.shape)
    
    # --------------------------------------------------------------------------
    # 3.1 运行当前精度下的标准 IID 实验
    # --------------------------------------------------------------------------
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, X_total_safe, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, grid=None
    )
    # 🛠️ 规范化修改：画图标签强制采用 Acc
    title_iid = f"Acc = {acc_str} (IID)"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, acc_str, None, xlim, ylim, title_iid, None)
    
    iid_total_records.append({
        'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # --------------------------------------------------------------------------
    # 3.2 运行当前精度下的三种 alpha_Dir Non-IID 实验
    # --------------------------------------------------------------------------
    alpha_dir_list = [1.0, 0.1, 0.01]
    dataset_dist_non = 'Non-IID'
    for alpha_dir in alpha_dir_list:
        X_dir, Y_dir, Yhat_dir, actual_sizes = dirichlet_logistic_regression_allocation(
            X_total_safe, Y_total, Yhat_total, alpha_dir, num_clients
        )
        
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, X_dir, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid=None
        )
        # 🛠️ 规范化修改：移除 Dirichlet 词汇，用纯学术符号 \alpha_{Dir} 和 Acc 呈现
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$ (Acc = {acc_str})"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, acc_str, alpha_dir, xlim, ylim, title_dir, None)
        
        non_iid_total_records.append({
            'dataset': dataset_name, 'target_accuracy': acc_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# Master CSV 全量指标合并落盘
csv_flat_dir = os.path.join('.', 'result', dataset_name, 'csv')
os.makedirs(csv_flat_dir, exist_ok=True)
if len(iid_total_records) > 0:
    pd.DataFrame(iid_total_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_IID_summary.csv'), index=False)
if len(non_iid_total_records) > 0:
    pd.DataFrame(non_iid_total_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_summary.csv'), index=False)


⚡ [当前进度] 正在全面计算精度级别 -> Acc = 0.1
labeled_ratio 0.3
分组： 1
带标签的样本量： 1442
不带标签的样本量： 3366
分组： 2
带标签的样本量： 1442
不带标签的样本量： 3366
分组： 3
带标签的样本量： 1442
不带标签的样本量： 3366
分组： 4
带标签的样本量： 1442
不带标签的样本量： 3366
分组： 5
带标签的样本量： 1442
不带标签的样本量： 3366
分组： 6
带标签的样本量： 1442
不带标签的样本量： 3366
分组： 7
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 8
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 9
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 10
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 11
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 12
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 13
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 14
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 15
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 16
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 17
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 18
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 19
带标签的样本量： 1442
不带标签的样本量： 3365
分组： 20
带标签的样本量： 1442
不带标签的样本量： 3365
带标签的样本量： 28840
不带标签的样本量： 67306

最终结果：
真实 theta: 0.16632510689756333
CPP intervals: [array([-0.02279092,  0.26807039]), array([-0.01254293,  0.25730729]), array([-0.05306589,  0.21630208]), array([-0.04348147,  0.23122001]), array([-0.027

In [4]:
# %%
# ==============================================================================
# 1. 固定准确率为 50% 的基准数据集调入与特征缩放
# ==============================================================================
fixed_acc_str = '0.5'
data_path = f'../data/{dataset_name}/{dataset_name}_acc_{fixed_acc_str}.npz'

print(f"正在读取固定 50% 准确率的目标基准数据: {data_path}")
data = np.load(data_path)
Y_total = data["Y"]
Yhat_total = data["Y_hat"] if "Y_hat" in data.files else data["Yhat"]

# 🛠️ 自适应特征标准化 (Z-score Scaling)
X_raw_fixed = data["X"].copy().astype(float)
for col in range(X_raw_fixed.shape[1]):
    col_std = X_raw_fixed[:, col].std()
    if col_std > 1e-2:
        X_raw_fixed[:, col] = (X_raw_fixed[:, col] - X_raw_fixed[:, col].mean()) / col_std
X_total_safe = X_raw_fixed + np.random.normal(0, 1e-3, X_raw_fixed.shape)

iid_ratio_records = []
non_iid_ratio_records = []
ratio_steps = [0.1, 0.2, 0.3, 0.4, 0.5]

# ==============================================================================
# 2. 纵向标签比例变动大循环核心
# ==============================================================================
for ratio in ratio_steps:
    ratio_str = f"{ratio:.1f}"
    sub_folder_name = f"ratio_{ratio_str}"  
    print(f"\n🚀 [实验进行中] 正在注入比例阶梯 -> labeled_ratio = {ratio_str}")
    
    # --------------------------------------------------------------------------
    # 4.1 变比率 IID 支线
    # --------------------------------------------------------------------------
    dataset_dist_iid = 'IID'
    num_ratio_iid = [1] * num_clients
    true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid = analyze_dataset(
        alpha, X_total_safe, Y_total, Yhat_total, dataset_dist_iid, num_ratio_iid, method, None, current_ratio=ratio
    )
    title_iid = f"$\\lambda = {ratio_str}$"
    plot_cpp(true_theta_iid, cpp_intervals_iid, ppi_ci_combined_iid, mean_cpp_iid, 
             dataset_name, dataset_dist_iid, fixed_acc_str, None, xlim, ylim, title_iid, sub_folder_name)
    
    iid_ratio_records.append({
        'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_iid, 'alpha_dir': 'None',
        'ppi_ci_lower': ppi_ci_combined_iid[0], 'ppi_ci_upper': ppi_ci_combined_iid[1], 'mean_cpp_lower': mean_cpp_iid[0], 'mean_cpp_upper': mean_cpp_iid[1]
    })
    
    # --------------------------------------------------------------------------
    # 4.2 变比率 Non-IID 支线
    # --------------------------------------------------------------------------
    dataset_dist_non = 'Non-IID'
    alpha_dir_list = [1.0, 0.1, 0.01]
    for alpha_dir in alpha_dir_list:
        # 🛠️ 核心修正：针对 10 万条庞大样本规模，将最低保底样本量提升至 1500，彻底消除极端非平衡下局部模型可分性坍塌
        X_dir, Y_dir, Yhat_dir, actual_sizes = dirichlet_logistic_regression_allocation(
            X_total_safe, Y_total, Yhat_total, alpha_dir, num_clients
        )
        
        true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir = analyze_dataset(
            alpha, X_dir, Y_dir, Yhat_dir, 'pre_split', actual_sizes, method, grid=None, current_ratio=ratio
        )
        title_dir = f"$\\alpha_{{Dir}} = {alpha_dir}$"
        plot_cpp(true_theta_dir, cpp_intervals_dir, ppi_ci_combined_dir, mean_cpp_dir, 
                 dataset_name, dataset_dist_non, fixed_acc_str, alpha_dir, xlim, ylim, title_dir, sub_folder_name)
        
        non_iid_ratio_records.append({
            'dataset': dataset_name, 'fixed_accuracy': fixed_acc_str, 'labeled_ratio': ratio_str, 'distribution': dataset_dist_non, 'alpha_dir': str(alpha_dir),
            'ppi_ci_lower': ppi_ci_combined_dir[0], 'ppi_ci_upper': ppi_ci_combined_dir[1], 'mean_cpp_lower': mean_cpp_dir[0], 'mean_cpp_upper': mean_cpp_dir[1]
        })

# 跨比率维度多合一 Ratio CSV 主表写出
if len(iid_ratio_records) > 0:
    pd.DataFrame(iid_ratio_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_IID_ratio_summary.csv'), index=False)
if len(non_iid_ratio_records) > 0:
    pd.DataFrame(non_iid_ratio_records).to_csv(os.path.join(csv_flat_dir, f'{dataset_name}_Non-IID_ratio_summary.csv'), index=False)

print(f"\n🎉 [数值安全防护注入成功] 变比率实验二已完美免疫所有边界崩溃，数据与规范 PDF 图表顺利产出！")

正在读取固定 50% 准确率的目标基准数据: ../data/diabetes-hypertension/diabetes-hypertension_acc_0.5.npz

🚀 [实验进行中] 正在注入比例阶梯 -> labeled_ratio = 0.1
labeled_ratio 0.1
分组： 1
带标签的样本量： 480
不带标签的样本量： 4328
分组： 2
带标签的样本量： 480
不带标签的样本量： 4328
分组： 3
带标签的样本量： 480
不带标签的样本量： 4328
分组： 4
带标签的样本量： 480
不带标签的样本量： 4328
分组： 5
带标签的样本量： 480
不带标签的样本量： 4328
分组： 6
带标签的样本量： 480
不带标签的样本量： 4328
分组： 7
带标签的样本量： 480
不带标签的样本量： 4327
分组： 8
带标签的样本量： 480
不带标签的样本量： 4327
分组： 9
带标签的样本量： 480
不带标签的样本量： 4327
分组： 10
带标签的样本量： 480
不带标签的样本量： 4327
分组： 11
带标签的样本量： 480
不带标签的样本量： 4327
分组： 12
带标签的样本量： 480
不带标签的样本量： 4327
分组： 13
带标签的样本量： 480
不带标签的样本量： 4327
分组： 14
带标签的样本量： 480
不带标签的样本量： 4327
分组： 15
带标签的样本量： 480
不带标签的样本量： 4327
分组： 16
带标签的样本量： 480
不带标签的样本量： 4327
分组： 17
带标签的样本量： 480
不带标签的样本量： 4327
分组： 18
带标签的样本量： 480
不带标签的样本量： 4327
分组： 19
带标签的样本量： 480
不带标签的样本量： 4327
分组： 20
带标签的样本量： 480
不带标签的样本量： 4327
带标签的样本量： 9600
不带标签的样本量： 86546

最终结果：
真实 theta: 0.1663168494412254
CPP intervals: [array([0.06783905, 0.91522504]), array([-0.03237201,  0.50676106]), array([-0.2